# 损失函数类

我们已经有了前向传播（层），现在需要一个衡量"预测有多不准"的工具——这就是**损失函数**。

损失函数是前向传播链路的**终点**：它接收预测值和标签值，输出一个表示"当前模型有多差"的数字，也就是**损失值**。损失值越小，说明预测越准；损失值越大，说明预测越差。

损失函数同时也是梯度计算链路的**起点**：后续章节中，梯度的计算将从损失值出发，沿着计算图反向传播。

---

在引入损失函数类之前，我们先来了解本章最重要的背景概念：**计算图**。

## 计算图

神经网络的计算并不是一步到位的。输入数据经过一层一层的神经元，每一步只做一个简单的局部运算，最终汇合到损失函数，得到损失值。这条数据流动的路径，就是**前向传播**。

为了能够计算梯度，我们需要在前向传播过程中，把数据流动的整个**拓扑结构**记录下来——哪个节点的输出，流向了哪个节点的输入。这个记录下来的拓扑结构，就称为**计算图**（Computational Graph）。

计算图中的每个**节点**是一个张量，可能是特征值、模型参数、中间计算结果，或者最终的损失值。节点之间的**连线**代表数据的流向和依赖关系。有了计算图，我们就能沿着它反向追溯，逐级计算每个参数的梯度。

因为计算图是在前向传播过程中**动态**构建的，所以称为**动态计算图**（Dynamic Computational Graph）。与之对应的是**静态计算图**（Static Computational Graph）——先把整个图的结构定义好，再送入数据运行。PyTorch 采用动态计算图，TensorFlow 早期采用静态计算图（新版本也支持动态模式）。我们将采用动态计算图。

``💡 本章我们先建立计算图的概念，并引入损失函数类完成前向传播链路。计算图的实际构建（记录节点间的依赖关系）将在下一章实现。``

In [21]:
from abc import ABC, abstractmethod
import numpy as np

## 基础架构

### 张量

In [22]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础层

In [23]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

**基础损失函数**（Base Loss）也是一个抽象类，和基础层的设计如出一辙。

它的核心方法叫 `loss`（而非 `forward`），因为损失函数在语义上不是"前向传播"的一部分，而是对预测结果的**评估**：接收预测值张量 `p` 和标签值张量 `y`，返回一个标量损失值张量。

In [24]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

## 数据

### 特征、标签

损失函数需要同时接收预测值和标签值才能计算误差，所以这里我们同时准备特征张量和标签张量：

In [25]:
feature = Tensor([28.1, 58.0])  # 温度 28.1°C，湿度 58%
label   = Tensor([165])         # 实际销量 165 支

## 模型

### 线性层

In [26]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        return Tensor(x.data @ self.weight.data.T + self.bias.data)

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 均方误差损失函数

我们定义的第一个损失函数是**均方误差**（Mean Squared Error，MSE），这是数值回归任务中最常用的损失函数。

它的计算方式是：对每个样本，求预测值与标签值之差的平方，再取所有样本的平均值：

$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - p_i)^2
$$

平方有两个作用：一是让正负误差都变成正数，不会相互抵消；二是放大较大的误差，使模型对偏差大的样本更敏感。

``💡 本章的 MSELoss 只计算损失值，暂时还不能支持反向传播。下一章我们会为它加上梯度计算函数，让它真正成为计算图的起点。``

In [27]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        return Tensor(np.mean(np.square(y.data - p.data)))

## 验证

### 建模

In [28]:
layer   = Linear(feature.shape[0], 1)
loss_fn = MSELoss()
print(layer)

Linear[weight(1, 2); bias(1,)]


### 推理

In [29]:
prediction = layer(feature)
print(f'prediction: {prediction}')

prediction: Tensor([43.05])


### 评估

In [30]:
loss = loss_fn(prediction, label)
print(f'loss: {loss}')

loss: Tensor(14871.802500000002)


损失值约为 `14871.8`。误差相当大，这正是一个未经训练的初始模型的正常表现。

至此，我们完成了前向传播链路的全部封装。

## 课后练习

实现**平均绝对误差**（MAELoss）。公式如下，试用同样的预测值和标签计算，并与 MSE 的结果比较：

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - p_i|
$$